# 06 M4: deep hedging under Heston with stock + variance swap (c = 0)
With the variance swap the market is complete: the Heston complete hedge (delta in the stock, vega / dL/dv in the swap) is the benchmark. M3 (stock only) is shown for comparison.

In [1]:
# notebooks live in notebooks/, the project modules in the repo root
import sys
if '..' not in sys.path:
    sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2

# standard libraries
import math

# third-party
import numpy as np
import pandas as pd
import torch
import seaborn as sns
from matplotlib import pyplot as plt

# project modules
import benchmarks as bm
from model import Policy
from market.simulators import simulate_heston, simulate_gbm, HestonParams
from market.pricing import bs_price, bs_delta, bs_gamma, heston_price
from features import nn_policy, ntb_policy
from train import train
from objectives import EntropicLoss, CVaRLoss


*Constants and the normalization dicts for M3 and M4 (`out_scale` lets the network output O(1) numbers while the swap position is ~350).*

In [10]:
# shared constants of the M4 experiments
K, T = 100.0, 30 / 365
params = HestonParams()                      # paper parameters

# feature / output normalization: one dict per model, used for training AND evaluation
feat_m3 = dict(s_scale=(params.b * T) ** 0.5, use_var=True, v_scale=params.b)
feat_kw = dict(**feat_m3, out_scale=torch.tensor([1.0, 100.0]))   # M4: variance-swap position ~ 100s

*Train the M4 network: 2 outputs (stock, variance swap), $\mathrm{CVaR}_{0.5}$.*

In [4]:
model_m4 = Policy(d_in=3, m_out=2)
out_m4 = train(model_m4, K, T, loss_fn=CVaRLoss(0.5), simulator=simulate_heston,
               sim_kwargs={"params": params}, n_instr=2, n_steps=3000, **feat_kw)

step: 0, loss: 3.497361421585083
step: 100, loss: 2.497992753982544
step: 200, loss: 2.021918296813965
step: 300, loss: 1.9857466220855713
step: 400, loss: 1.9874053001403809
step: 500, loss: 1.9678138494491577
step: 600, loss: 1.9798029661178589
step: 700, loss: 1.9594403505325317
step: 800, loss: 1.95615553855896
step: 900, loss: 1.9606064558029175
step: 1000, loss: 1.9433685541152954
step: 1100, loss: 1.9434221982955933
step: 1200, loss: 1.9413398504257202
step: 1300, loss: 1.9550033807754517
step: 1400, loss: 1.9379464387893677
step: 1500, loss: 1.9461709260940552
step: 1600, loss: 1.936611294746399
step: 1700, loss: 1.9427882432937622
step: 1800, loss: 1.9527746438980103
step: 1900, loss: 1.9403150081634521
step: 2000, loss: 1.9452959299087524
step: 2100, loss: 1.9369686841964722
step: 2200, loss: 1.9327666759490967
step: 2300, loss: 1.9349452257156372
step: 2400, loss: 1.9400372505187988
step: 2500, loss: 1.938510775566101
step: 2600, loss: 1.9294965267181396
step: 2700, loss: 1.

*Train an M3 network (stock only) on the same setting, for comparison.*

In [5]:
model_m3 = Policy(d_in = 3, m_out = 1)
out_m3 = train(model_m3, K, T, loss_fn=CVaRLoss(0.5), simulator=simulate_heston,
               sim_kwargs={"params": params}, n_steps=3000, **feat_m3)

step: 0, loss: 3.4220948219299316
step: 100, loss: 2.609332799911499
step: 200, loss: 2.467740774154663
step: 300, loss: 2.5183253288269043
step: 400, loss: 2.491927146911621
step: 500, loss: 2.5924150943756104
step: 600, loss: 2.525196075439453
step: 700, loss: 2.562157154083252
step: 800, loss: 2.5149691104888916
step: 900, loss: 2.474581241607666
step: 1000, loss: 2.481839656829834
step: 1100, loss: 2.4525716304779053
step: 1200, loss: 2.477036476135254
step: 1300, loss: 2.5127038955688477
step: 1400, loss: 2.500128984451294
step: 1500, loss: 2.5406787395477295
step: 1600, loss: 2.5354526042938232
step: 1700, loss: 2.554232120513916
step: 1800, loss: 2.520927906036377
step: 1900, loss: 2.546093463897705
step: 2000, loss: 2.5357871055603027
step: 2100, loss: 2.533820152282715
step: 2200, loss: 2.500283718109131
step: 2300, loss: 2.5084927082061768
step: 2400, loss: 2.526909112930298
step: 2500, loss: 2.5012807846069336
step: 2600, loss: 2.564207077026367
step: 2700, loss: 2.503938198

*Fixed Heston test set (seed 0) and the CVaR evaluation helper (chunked, since `heston_greeks` is memory-heavy).*

In [9]:
with torch.no_grad():
    torch.manual_seed(0)
    test = simulate_heston(100_000, params=params)      # fixed Heston test set
    
def eval_cvar(pol, alpha, n_instr=1, chunk=10_000):
    """Price = CVaR_alpha of the loss on the fixed Heston test set, plus P&L summary.

    n_instr=1: hedge with the stock only (M3 policies)
    n_instr=2: hedge with stock + variance swap (M4 policies, complete hedge)
    The test set is processed in chunks of `chunk` paths to limit memory
    (heston_greeks builds a (paths x 512) complex matrix at every date).
    """
    S_all = test["S"][:, :, 0] if n_instr == 1 else test["S"][:, :, :n_instr]
    pnls = []
    with torch.no_grad():
        for i in range(0, S_all.shape[0], chunk):
            pnls.append(bm.hedge_pnl(S_all[i:i + chunk], pol, K, T,
                                     V=test["V"][i:i + chunk]))     # p0 = 0
    pnl = torch.cat(pnls)
    p0 = bm.summarize(pnl, alpha)["CVaR"]
    return {"p0": p0, **bm.summarize(pnl + p0, alpha)}

*$\mathrm{CVaR}_{0.5}$ prices: M4 NN vs Heston complete hedge vs M3 NN vs BS $\sqrt{V_k}$. M4 NN should be close to the complete hedge and far below the stock-only strategies.*

In [11]:
rows = {
    "M4 NN (stock + var swap)": eval_cvar(nn_policy(model_m4, K, T, **feat_kw), 0.5, n_instr=2),
    "Heston complete hedge":    eval_cvar(bm.heston_complete_policy(K, params), 0.5, n_instr=2),
    "M3 NN (stock only)":       eval_cvar(nn_policy(model_m3, K, T, **feat_m3), 0.5),
    "BS √V_k (stock only)":     eval_cvar(bm.bs_delta_stovol_policy(K), 0.5),
}
pd.DataFrame(rows).T

,p0,mean,std,VaR,CVaR
M4 NN (stock + var swap),1.946062,0.252327,0.405842,-0.262486,1.584625e-07
Heston complete hedge,1.938121,0.242941,0.383523,-0.260414,8.392334e-09
M3 NN (stock only),2.514788,0.817174,1.195009,-1.085457,1.184082e-07
BS √V_k (stock only),2.670661,0.976190,1.349290,-1.455262,-2.822876e-07
